# Setup storage layout

This notebook creates the shared storage setup for the workspace.

- Create the shared bronze/silver/gold schemas.
- Create the shared landing volume for source archives.
- Leave dataset-specific paths and DDL to the dataset setup notebooks.

In [ ]:
%run ./_setup_env


In [ ]:
from data_platform.setup import load_yaml_config

STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
LANDING_ROOT = STORAGE_CONFIG["landing_root"]
STORAGE_ROOT = STORAGE_CONFIG["storage_root"]


MEDALLION_LAYERS = ["bronze", "silver", "gold"]


In [ ]:
dbutils.fs.mkdirs(LANDING_ROOT)
dbutils.fs.mkdirs(f"{LANDING_ROOT.rstrip('/')}/archives")

for layer in MEDALLION_LAYERS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {layer}")
    dbutils.fs.mkdirs(f"{STORAGE_ROOT.rstrip('/')}/{layer}")

display(spark.sql("SHOW SCHEMAS"))
display(dbutils.fs.ls(LANDING_ROOT))
display(dbutils.fs.ls(STORAGE_ROOT))

In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.ingestion_runs (
      ingestion_run_id STRING,
      dataset_name STRING,
      source_version STRING,
      started_at TIMESTAMP,
      finished_at TIMESTAMP,
      status STRING,
      records_seen INT,
      records_written INT
    )
    USING DELTA
    """
)

display(spark.sql("SHOW TABLES IN bronze"))


## Storage conventions

- Landing volume convention: `archives/<source-archive>.zip`
- Bronze storage convention: `isic_2019/...`
- Silver storage convention: `silver/<dataset>/<version>/...`
- Gold storage convention: `gold/<product>/<version>/...`


## Dataset-specific setup

Dataset notebooks should create dataset-specific paths and dataset-specific tables.

Current example:

- `notebooks/isic_2019/10_bronze_setup.ipynb`

That notebook should load dataset config, create dataset-specific DDL, and manage dataset-specific folder paths.